# Moving the IPSL sources to OSN

Companion to `notebooks/reference-generation/local-ref-generation-ipsl.ipynb`,
which proves the *mechanism* against local files. A local `file://` store is not shareable: the
chunk manifests contain absolute paths on one laptop. To publish, the **source NetCDF**
has to live somewhere addressable, and the store has to be rebuilt so its manifests
point there.

⚠️ **Nothing in this section has been executed.** The upload is the user's to run.

## Step 1 — credentials (1Password)

The OSN keys are read from 1Password with the `op` CLI, exactly as in
`notebooks/reference-generation/build_static_examples.ipynb`:

```python
import subprocess

def op_read(ref):
    return subprocess.check_output(["op", "read", ref]).decode().strip()

key    = op_read("op://Work/z6baienaiyhiexztlbbonbeaka/Read-Write/Access_Key")
secret = op_read("op://Work/z6baienaiyhiexztlbbonbeaka/Read-Write/Secret_Access_Key")
```

Requires the 1Password desktop app unlocked with CLI integration enabled.

## Step 2 — upload the four NetCDF files

Bucket and endpoint follow the existing convention
(`s3://leap-pangeo-pipeline`, `https://nyu1.osn.mghpcc.org`). Source data goes under a
`source-data/` prefix so it is not confused with the `icechunk-v2/` stores.

```bash
export AWS_ACCESS_KEY_ID=$(op read "op://Work/z6baienaiyhiexztlbbonbeaka/Read-Write/Access_Key")
export AWS_SECRET_ACCESS_KEY=$(op read "op://Work/z6baienaiyhiexztlbbonbeaka/Read-Write/Secret_Access_Key")

aws s3 cp \
  /Users/juliusbusecke/Code/cmip7-virtualization/notebooks/reference-generation/data/ \
  s3://leap-pangeo-pipeline/cmip7-virtualization/source-data/ipsl-cmip7-test/ \
  --recursive --exclude "*" --include "*.nc" \
  --endpoint-url https://nyu1.osn.mghpcc.org

# verify
aws s3 ls s3://leap-pangeo-pipeline/cmip7-virtualization/source-data/ipsl-cmip7-test/ \
  --endpoint-url https://nyu1.osn.mghpcc.org
```

~1.6 GB, so allow time. The bucket is public-read — the existing download snippets use
`--no-sign-request` — so the virtual-chunk container below is anonymous.

## Step 3 — rebuild the stores against OSN

`virtualize_from_urls` grew an `s3_endpoint_url` argument for exactly this: an
S3-compatible gateway needs the endpoint plus path-style addressing, which neither the
esgf-world nor the AWS path supplies. The same endpoint has to be threaded into the
virtual-chunk container via `vccs_from_registry(..., s3_endpoint_url=...)`, or the
metadata will open and every chunk read will fail.

Set `RUN_OSN = True` **after** the upload completes.

In [ ]:
import subprocess
from pathlib import Path

import icechunk as ic

import cmip7_virtualization as cv
from cmip7_virtualization.storage import (
    authorize_prefixes_from_registry,
    vccs_from_registry,
)
from cmip7_virtualization.virtualize import virtualize_from_urls

# The NetCDF files live in the MAIN checkout, not in this worktree, and are never
# copied. Same absolute path as the local notebook.
DATA = Path(
    "/Users/juliusbusecke/Code/cmip7-virtualization/notebooks/reference-generation/data"
)

GROUPS = {
    "ficeberg": sorted(DATA.glob("ficeberg_*.nc")),  # multi-file concat, 3hr
    "thetao": sorted(DATA.glob("thetao_*.nc")),  # single file, mon, 4-D
    "areacello": sorted(DATA.glob("areacello_*.nc")),  # fx, no time dimension
}

for name, paths in GROUPS.items():
    assert paths, f"no files found for {name} in {DATA}"
print({k: [p.name for p in v] for k, v in GROUPS.items()})

In [20]:
# ----------------------------- FILL IN AFTER UPLOAD -----------------------------
RUN_OSN = False  # <-- flip to True once step 2 has completed

OSN_BUCKET = "leap-pangeo-pipeline"  # existing bucket
OSN_SOURCE_PREFIX = "cmip7-virtualization/source-data/ipsl-cmip7-test"  # step 2 target
OSN_STORE_PREFIX = (
    "cmip7-virtualization/icechunk-v2/ipsl-cmip7-test"  # where stores land
)
# 1Password item holding the OSN Read-Write keys (same item as build_static_examples).
OP_ACCESS_KEY = "op://Work/z6baienaiyhiexztlbbonbeaka/Read-Write/Access_Key"
OP_SECRET_KEY = "op://Work/z6baienaiyhiexztlbbonbeaka/Read-Write/Secret_Access_Key"
# --------------------------------------------------------------------------------

OSN_ENDPOINT = cv.OSN_ENDPOINT_URL  # https://nyu1.osn.mghpcc.org
osn_urls = {
    name: [f"s3://{OSN_BUCKET}/{OSN_SOURCE_PREFIX}/{p.name}" for p in paths]
    for name, paths in GROUPS.items()
}
osn_urls

{'ficeberg': ['s3://leap-pangeo-pipeline/cmip7-virtualization/source-data/ipsl-cmip7-test/ficeberg_tavg-ol-hxy-sea_3hr_glb_g112_IPSLCM6-ESMCO2_piControl_r1i1p1f1_185001010130-185004302230.nc',
  's3://leap-pangeo-pipeline/cmip7-virtualization/source-data/ipsl-cmip7-test/ficeberg_tavg-ol-hxy-sea_3hr_glb_g112_IPSLCM6-ESMCO2_piControl_r1i1p1f1_185005010130-185008312230.nc'],
 'thetao': ['s3://leap-pangeo-pipeline/cmip7-virtualization/source-data/ipsl-cmip7-test/thetao_tavg-ol-hxy-sea_mon_glb_g112_IPSLCM6-ESMCO2_piControl_r1i1p1f1_185001-185912.nc'],
 'areacello': ['s3://leap-pangeo-pipeline/cmip7-virtualization/source-data/ipsl-cmip7-test/areacello_ti-u-hxy-u_fx_glb_g112_IPSLCM6-ESMCO2_piControl_r1i1p1f1.nc']}

In [21]:
def op_read(ref):
    return subprocess.check_output(["op", "read", ref]).decode().strip()


def build_on_osn(name):
    """Virtualize the OSN-hosted copies and write the store back to OSN.

    Untested — the sources do not exist yet. Differences from the local path:
      * sources are read anonymously through the OSN endpoint (path-style addressing),
      * the virtual-chunk container must carry the same endpoint,
      * the repository itself is written with the Read-Write keys from 1Password.
    """
    vds, registry = virtualize_from_urls(
        osn_urls[name], s3_endpoint_url=OSN_ENDPOINT, s3_region="us-east-1"
    )

    config = ic.RepositoryConfig.default()
    for vcc in vccs_from_registry(
        registry, s3_endpoint_url=OSN_ENDPOINT, s3_region="us-east-1"
    ):
        config.set_virtual_chunk_container(vcc)

    repo = ic.Repository.open_or_create(
        storage=cv.osn_storage(
            bucket=OSN_BUCKET,
            prefix=f"{OSN_STORE_PREFIX}/{name}/",
            access_key_id=op_read(OP_ACCESS_KEY),
            secret_access_key=op_read(OP_SECRET_KEY),
        ),
        config=config,
        authorize_virtual_chunk_access=authorize_prefixes_from_registry(registry),
    )
    session = repo.writable_session("main")
    vds.vz.to_icechunk(session.store)
    snapshot = session.commit(f"virtual references for {name} (OSN sources)")
    repo.save_config()
    return snapshot


if RUN_OSN:
    for name in GROUPS:
        print(name, build_on_osn(name))
else:
    print("RUN_OSN is False — upload the sources (step 2) first.")

RUN_OSN is False — upload the sources (step 2) first.
